# Lab 04.6 — Color Position on Zybo LEDs

## Guided experiment — about 10 minutes

Use the working color tracker to light one LED according to the object's position. All code is supplied: **no state-machine design, button handling or large programming assignment is required**.

| Position in the displayed image | LED |
|---|---|
| LEFT | LD0 |
| CENTER | LD1 |
| RIGHT | LD2 |
| No target (LOST) | LD3 |

Vision runs on the ARM processor; GPIO in the base overlay drives the LEDs. This uses the same overlay and AXI GPIO initialization as Lab01 and Lab03.

## Camera setup — provided code

Run this helper cell once in this notebook. It uses the previously verified **V4L2 / `/dev/video0` → 20 warm-up frames → capture → inline JPEG display → release** sequence. The requested resolution is 640×480; use `frame.shape` to check the actual resolution.

The `with camera_session()` block releases the camera through `finally`, including on Python errors or a normal Jupyter interrupt. Only one notebook/kernel should use the camera at a time. If Linux assigns a different capture device, change **CAMERA_DEVICE in each notebook you use**; notebook variables are not shared.

The helper is supplied; you do not need to implement it. Time limits are checked between reads: a USB driver call that blocks can delay an interrupt or timeout.

In [ ]:
import cv2
import numpy as np
import time
from contextlib import contextmanager
from IPython.display import display, Image, clear_output

CAMERA_DEVICE = "/dev/video0"
WARMUP_FRAMES = 20

def read_frame(cap):
    ok, frame = cap.read()
    if not ok or frame is None or frame.size == 0:
        raise RuntimeError(
            "No camera frame. Check the USB connection and close other camera notebooks."
        )
    return frame

@contextmanager
def camera_session():
    """Provided helper: open, warm up, and always release the camera."""
    cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
    try:
        if not cap.isOpened():
            raise RuntimeError(
                "Cannot open " + CAMERA_DEVICE +
                ". Check the device and shut down other kernels using the camera."
            )
        # Request a modest resolution; the driver may negotiate another one.
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        warmup_start = time.monotonic()
        for _ in range(WARMUP_FRAMES):
            read_frame(cap)  # Discard frames while exposure settles.
            if time.monotonic() - warmup_start > 10:
                raise RuntimeError("Camera warm-up is too slow. Check the USB connection.")
        yield cap
    finally:
        cap.release()
        print("Camera released.")

def show_bgr(frame, replace=False):
    ok, jpg = cv2.imencode(".jpg", frame)
    if not ok:
        raise RuntimeError("JPEG encoding failed.")
    if replace:
        clear_output(wait=True)
    display(Image(data=jpg.tobytes()))


## Initialize the LEDs

Run once before the demonstration. As in Lab01, `Overlay("base.bit")` loads the board's existing base overlay. The four bits of `leds_gpio.channel1` control LD0…LD3. Do not load another overlay from a different notebook while this experiment is running.

In [ ]:
from pynq import Overlay

ol = Overlay("base.bit")
leds = ol.leds_gpio.channel1
leds.setdirection("out")
leds.setlength(4)

def all_leds_off():
    leds.write(0, 0b1111)

all_leds_off()
print("LED GPIO initialized, as in Lab01.")


In [ ]:
TARGET = "GREEN"

RANGES = {
    "BLUE":   [((100, 100, 60), (130, 255, 255))],
    "GREEN":  [((35, 80, 60), (85, 255, 255))],
    "YELLOW": [((20, 100, 100), (34, 255, 255))],
    "RED":    [((0, 100, 80), (10, 255, 255)),
               ((170, 100, 80), (179, 255, 255))]
}

def color_mask(hsv, color_name):
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)

    for lower, upper in RANGES[color_name]:
        mask |= cv2.inRange(hsv, np.array(lower), np.array(upper))

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

def horizontal_region(cx, width):
    if cx < width / 3:
        return "LEFT"
    elif cx < 2 * width / 3:
        return "CENTER"
    else:
        return "RIGHT"

def set_led_for_region(region):
    patterns = {"LEFT": 0b0001, "CENTER": 0b0010,
                "RIGHT": 0b0100, "LOST": 0b1000}
    leds.write(patterns[region], 0b1111)


## Run the LED demonstration

Hold a large, saturated colored object in front of the camera. The session stops after at most 180 processed frames or approximately 15 seconds after warm-up. These are limits checked between reads, not a guaranteed driver timeout. Use **Kernel → Interrupt** to stop early; rerun this cell for another session.

640×480 is requested. If you use another resolution, the fixed `MIN_AREA` threshold may need adjustment. `time.sleep(0.03)` is only a short pause; it does not guarantee a frame rate. Display and processing also take time.

In [ ]:
MIN_AREA = 1200  # Contour area threshold; pixels squared.
MAX_FRAMES = 180
MAX_SECONDS = 15

try:
    with camera_session() as cap:
        start = time.monotonic()
        for _ in range(MAX_FRAMES):
            if time.monotonic() - start >= MAX_SECONDS:
                break
            frame = read_frame(cap)

            hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
            mask = color_mask(hsv, TARGET)

            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            valid = [c for c in contours if cv2.contourArea(c) >= MIN_AREA]

            region = "LOST"

            if valid:
                cnt = max(valid, key=cv2.contourArea)
                x, y, w, h = cv2.boundingRect(cnt)

                M = cv2.moments(cnt)
                if M["m00"] != 0:
                    cx = int(M["m10"] / M["m00"])
                    cy = int(M["m01"] / M["m00"])

                    region = horizontal_region(cx, frame.shape[1])

                    cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 255, 255), 2)
                    cv2.circle(frame, (cx, cy), 6, (255, 255, 255), -1)

            set_led_for_region(region)

            cv2.putText(frame, f"TARGET: {TARGET}  STATE: {region}",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                        0.7, (255, 255, 255), 2)

            show_bgr(frame, replace=True)

            time.sleep(0.03)

finally:
    all_leds_off()
    print("LEDs switched off.")


## Short exercise

1. Run the demonstration with a green object.
2. Move the object through LEFT, CENTER and RIGHT in the displayed image; observe LD0, LD1 and LD2.
3. Remove the object and check that LD3 lights up while the session is running.
4. Let the session finish and confirm that all LEDs turn off.

If time permits, change `TARGET` to `"BLUE"`, rerun its definition cell, and try a blue object. No additional extension is required.

**Record:** Did all four LED states work? Describe one situation in which the color detection was unreliable.